# 12 策略分类地图：从主线策略到经典策略

## 12.1 本章要解决什么问题

前面 11 章已经把 ETF 多因子策略从数据、因子、组合、回测、报告和信号输出跑通。继续写更多策略前，先建立一张分类地图，避免把“双均线”“布林带”“动量”“多因子”当成互不相干的名字。

## 12.2 位置、输入与输出

- 上一章：主线 ETF 多因子流水线已经可复现运行，并能输出目标权重与交易建议。
- 本章输入：样例 ETF 池、主线策略配置、前面已经形成的回测口径。
- 本章输出：策略分类轴、主线策略标签、经典策略预览标签、一个练习答案表。
- 下一章：直接实现双均线、布林带和横截面动量，并复用 `lib` 做回测和报告。

本章代码较少，但每一段都会生成可检查的表格。


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Cannot find pyquant-roadmap project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import yaml

from lib.data import load_sample_assets, load_sample_prices
from lib.paths import CONFIG_DIR, RESULTS_DIR

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

OUTPUT_DIR = RESULTS_DIR / "chapter12_strategy_taxonomy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assets = load_sample_assets()
prices = load_sample_prices()
strategy_cfg = yaml.safe_load((CONFIG_DIR / "strategy_demo.yml").read_text(encoding="utf-8"))

sample_context = pd.DataFrame(
    [
        ("ETF 数量", assets["code"].nunique()),
        ("样例数据开始", prices["date"].min().date()),
        ("样例数据结束", prices["date"].max().date()),
        ("主线调仓频率", strategy_cfg["rebalance_freq"]),
        ("主线 TopN", strategy_cfg["top_n"]),
        ("回测成本 bps", strategy_cfg["cost_bps"]),
    ],
    columns=["项目", "值"],
)
sample_context


,项目,值
0,ETF 数量,4
1,样例数据开始,2021-01-04
2,样例数据结束,2023-12-29
3,主线调仓频率,M
4,主线 TopN,3
5,回测成本 bps,8


## 12.3 六个分类维度

分类不是给策略贴一个唯一名字，而是同时回答六个问题：信号从哪里来、持有多久、交易哪些资产、仓位能不能做空、信号怎样变成权重、回测假设是否可信。


In [2]:
taxonomy_axes = pd.DataFrame(
    [
        (
            "信号来源",
            "为什么交易",
            "趋势/动量；均值回归；统计套利/相对价值；事件驱动；基本面/因子",
            "决定要计算什么特征，以及哪里最容易出现未来函数。",
        ),
        (
            "持仓周期",
            "信号多久换一次",
            "日内/高频；日频/周频；月频/低频",
            "决定噪声水平、交易成本、撮合粒度和样本数量。",
        ),
        (
            "资产范围",
            "交易一个资产还是一组资产",
            "单资产；多资产横截面；配对/篮子；跨资产",
            "决定数据对齐、缺失值处理和可交易性检查。",
        ),
        (
            "仓位方向",
            "仓位可以取哪些方向",
            "多头/现金；多空；市场中性；杠杆",
            "决定是否需要借券、保证金、净敞口和风险预算。",
        ),
        (
            "组合构建",
            "信号怎样变成目标权重",
            "择时满仓/空仓；TopN 等权；分数加权；风险约束/优化",
            "决定回测输入是单列仓位、权重矩阵还是优化器输出。",
        ),
        (
            "回测假设",
            "模拟是否接近真实可交易",
            "下一期成交；成本/滑点；再平衡日；容量；成分可得性",
            "决定净收益和报告可信度，尤其要写清 signal 和 position 的时点。",
        ),
    ],
    columns=["维度", "关键问题", "常见标签", "对回测的影响"],
)
taxonomy_axes


,维度,关键问题,常见标签,对回测的影响
0,信号来源,为什么交易,趋势/动量；均值回归；统计套利/相对价值；事件驱动；基本面/因子,决定要计算什么特征，以及哪里最容易出现未来函数。
1,持仓周期,信号多久换一次,日内/高频；日频/周频；月频/低频,决定噪声水平、交易成本、撮合粒度和样本数量。
2,资产范围,交易一个资产还是一组资产,单资产；多资产横截面；配对/篮子；跨资产,决定数据对齐、缺失值处理和可交易性检查。
3,仓位方向,仓位可以取哪些方向,多头/现金；多空；市场中性；杠杆,决定是否需要借券、保证金、净敞口和风险预算。
4,组合构建,信号怎样变成目标权重,择时满仓/空仓；TopN 等权；分数加权；风险约束/优化,决定回测输入是单列仓位、权重矩阵还是优化器输出。
5,回测假设,模拟是否接近真实可交易,下一期成交；成本/滑点；再平衡日；容量；成分可得性,决定净收益和报告可信度，尤其要写清 signal 和 position 的时点。


## 12.4 把主线 ETF 多因子策略放进地图

主线案例不是一个孤立策略，它同时落在“多因子信号、低频持有、多 ETF 横截面、多头 TopN 等权”这几个坐标上。


In [3]:
factor_text = " + ".join(f"{name}({weight:.0%})" for name, weight in strategy_cfg["factor_weights"].items())
asset_names = assets.assign(label=assets["code"] + " " + assets["name"])["label"].tolist()

mainline_map = pd.DataFrame(
    [
        ("信号来源", f"多因子综合打分：{factor_text}", "先算特征，再把多个特征合成一个排序分数。"),
        ("持仓周期", f"{strategy_cfg['rebalance_freq']} 再平衡，偏低频", "减少噪声和交易成本，更适合个人学习闭环。"),
        ("资产范围", f"{len(asset_names)} 只 ETF 横截面：" + "；".join(asset_names), "每天比较同一池子里的相对吸引力。"),
        ("仓位方向", "多头/现金，不做空、不加杠杆", "先避开借券、保证金和爆仓路径。"),
        ("组合构建", f"Top{strategy_cfg['top_n']} 等权", "把排序结果变成目标权重矩阵。"),
        ("回测假设", f"目标权重下一期生效，扣 {strategy_cfg['cost_bps']} bps 成本", "和前面章节的权重型回测口径保持一致。"),
    ],
    columns=["维度", "主线 ETF 多因子标签", "含义"],
)
mainline_map


,维度,主线 ETF 多因子标签,含义
0,信号来源,多因子综合打分：momentum_60(45%) + low_vol_20(35%) + m...,先算特征，再把多个特征合成一个排序分数。
1,持仓周期,M 再平衡，偏低频,减少噪声和交易成本，更适合个人学习闭环。
2,资产范围,4 只 ETF 横截面：510300 沪深300ETF；510500 中证500ETF；15...,每天比较同一池子里的相对吸引力。
3,仓位方向,多头/现金，不做空、不加杠杆,先避开借券、保证金和爆仓路径。
4,组合构建,Top3 等权,把排序结果变成目标权重矩阵。
5,回测假设,目标权重下一期生效，扣 8 bps 成本,和前面章节的权重型回测口径保持一致。


## 12.5 用同一套标签预览经典策略

下一章会写三种代表作。这里先只看它们的坐标，等到 notebook 13 再把坐标落成函数和权重矩阵。


In [4]:
strategy_cards = pd.DataFrame(
    [
        {
            "策略": "主线 ETF 多因子 TopN",
            "信号来源": "基本面/因子风格的技术化示例：动量、低波动、均线乖离",
            "持仓周期": "月频/低频",
            "资产范围": "多 ETF 横截面",
            "仓位方向": "多头/现金",
            "组合构建": "TopN 等权",
            "关键回测假设": "信号滞后一周期；扣成本；固定样例 ETF 池",
            "下一章对应": "前 11 章主线案例",
        },
        {
            "策略": "双均线趋势",
            "信号来源": "时间序列趋势/动量",
            "持仓周期": "日频信号，可低频复盘",
            "资产范围": "单 ETF",
            "仓位方向": "多头/现金",
            "组合构建": "择时：短均线高于长均线则持有，否则空仓",
            "关键回测假设": "复权价格；信号下一期生效；震荡市成本敏感",
            "下一章对应": "13 策略一",
        },
        {
            "策略": "布林带均值回归",
            "信号来源": "价格偏离后的均值回归",
            "持仓周期": "日频/波段",
            "资产范围": "单 ETF",
            "仓位方向": "多头/现金",
            "组合构建": "状态机：跌破下轨买入，回到中轨退出",
            "关键回测假设": "强趋势会失效；需要成本、止损或持仓天数约束",
            "下一章对应": "13 策略二",
        },
        {
            "策略": "横截面动量 TopN",
            "信号来源": "资产之间的相对强弱",
            "持仓周期": "月频再平衡",
            "资产范围": "多 ETF 横截面",
            "仓位方向": "多头/现金",
            "组合构建": "过去收益排名 TopN 等权",
            "关键回测假设": "统一调仓日；处理缺失值；扣换手成本",
            "下一章对应": "13 策略三",
        },
    ]
)
strategy_cards


,策略,信号来源,持仓周期,资产范围,仓位方向,组合构建,关键回测假设,下一章对应
0,主线 ETF 多因子 TopN,基本面/因子风格的技术化示例：动量、低波动、均线乖离,月频/低频,多 ETF 横截面,多头/现金,TopN 等权,信号滞后一周期；扣成本；固定样例 ETF 池,前 11 章主线案例
1,双均线趋势,时间序列趋势/动量,日频信号，可低频复盘,单 ETF,多头/现金,择时：短均线高于长均线则持有，否则空仓,复权价格；信号下一期生效；震荡市成本敏感,13 策略一
2,布林带均值回归,价格偏离后的均值回归,日频/波段,单 ETF,多头/现金,状态机：跌破下轨买入，回到中轨退出,强趋势会失效；需要成本、止损或持仓天数约束,13 策略二
3,横截面动量 TopN,资产之间的相对强弱,月频再平衡,多 ETF 横截面,多头/现金,过去收益排名 TopN 等权,统一调仓日；处理缺失值；扣换手成本,13 策略三


## 12.6 简单规则标签器

真实研究里不能只靠关键词分类，但一个小规则表能帮助读者先形成检查清单：读到一段策略描述时，先抽取关键词，再补人工判断。


In [5]:
TAG_RULES = {
    "信号来源": [
        ("趋势/动量", ["均线", "突破", "动量", "过去收益", "强者"]),
        ("均值回归", ["布林", "偏离", "回归", "下轨", "标准差", "价差"]),
        ("统计套利/相对价值", ["配对", "协整", "相对价值", "市场中性"]),
        ("事件驱动", ["财报", "公告", "并购", "回购", "指数调样"]),
        ("基本面/因子", ["因子", "估值", "质量", "成长", "低波动", "打分"]),
    ],
    "持仓周期": [
        ("日内/高频", ["日内", "分钟", "tick", "盘口"]),
        ("日频/波段", ["每日", "每天", "日频", "收盘", "下一交易日"]),
        ("周频", ["每周", "周频"]),
        ("月频/低频", ["每月", "月末", "月频", "月底"]),
    ],
    "资产范围": [
        ("配对/篮子", ["两只", "配对", "价差", "协整"]),
        ("多资产横截面", ["多只", "ETF 池", "ETF池", "横截面", "Top", "排名"]),
        ("单资产", ["一只", "单只", "沪深300ETF", "510300"]),
    ],
    "仓位方向": [
        ("杠杆", ["杠杆", "保证金"]),
        ("多空/市场中性", ["做空", "空头", "多空", "市场中性"]),
        ("多头/现金", ["只做多", "多头", "买入", "清仓", "现金"]),
    ],
    "组合构建": [
        ("TopN 等权", ["Top", "前", "排名", "等权"]),
        ("择时满仓/空仓", ["均线", "布林", "买入", "卖出", "清仓"]),
        ("分数加权", ["打分", "分数", "因子权重"]),
        ("风险约束/优化", ["风险平价", "最小方差", "优化"]),
    ],
    "回测假设": [
        ("信号下一期生效", ["下一交易日", "下一期", "shift", "滞后"]),
        ("扣成本/滑点", ["成本", "滑点", "bps", "手续费"]),
        ("再平衡日明确", ["每月", "月末", "月底", "每周"]),
        ("成分与数据可得性", ["ETF 池", "成分", "可得", "缺失"]),
    ],
}


def classify_description(text: str) -> pd.DataFrame:
    text_lower = text.lower()
    rows = []
    for axis, rules in TAG_RULES.items():
        matched = []
        evidence = []
        for tag, keywords in rules:
            hits = [kw for kw in keywords if kw.lower() in text_lower]
            if hits:
                matched.append(tag)
                evidence.append(f"{tag}: " + "/".join(hits))
        rows.append(
            {
                "维度": axis,
                "标签": "；".join(dict.fromkeys(matched)) if matched else "未识别，需要人工补充",
                "触发线索": "；".join(evidence) if evidence else "-",
            }
        )
    return pd.DataFrame(rows)


example_description = "每月月底在 4 只 ETF 池中计算过去 60 日动量，选 Top2 等权买入；只做多，下一交易日成交，扣 8 bps 成本。"
classify_description(example_description)


,维度,标签,触发线索
0,信号来源,趋势/动量,趋势/动量: 动量
1,持仓周期,日频/波段；月频/低频,日频/波段: 下一交易日；月频/低频: 每月/月底
2,资产范围,多资产横截面,多资产横截面: ETF 池/Top
3,仓位方向,多头/现金,多头/现金: 只做多/买入
4,组合构建,TopN 等权；择时满仓/空仓,TopN 等权: Top/等权；择时满仓/空仓: 买入
5,回测假设,信号下一期生效；扣成本/滑点；再平衡日明确；成分与数据可得性,信号下一期生效: 下一交易日；扣成本/滑点: 成本/bps；再平衡日明确: 每月/月底；成分...


## 12.7 练习：给一个陌生策略打坐标

先阅读下面这段描述，尝试填写“你的标签”。下一格会给出基于规则表的参考答案。


In [6]:
exercise_description = "一只沪深300ETF 的收盘价跌破 20 日均线下方 2 倍标准差时买入，回到中轨时清仓；只做多，每天收盘后生成信号，下一交易日成交，扣 8 bps 成本。"

reader_guess = pd.DataFrame(
    {
        "维度": list(TAG_RULES.keys()),
        "你的标签": ["" for _ in TAG_RULES],
    }
)

print(exercise_description)
reader_guess


一只沪深300ETF 的收盘价跌破 20 日均线下方 2 倍标准差时买入，回到中轨时清仓；只做多，每天收盘后生成信号，下一交易日成交，扣 8 bps 成本。


,维度,你的标签
0,信号来源,
1,持仓周期,
2,资产范围,
3,仓位方向,
4,组合构建,
5,回测假设,


In [7]:
exercise_answer = classify_description(exercise_description)
exercise_answer


,维度,标签,触发线索
0,信号来源,趋势/动量；均值回归,趋势/动量: 均线；均值回归: 标准差
1,持仓周期,日频/波段,日频/波段: 每天/收盘/下一交易日
2,资产范围,单资产,单资产: 一只/沪深300ETF
3,仓位方向,多头/现金,多头/现金: 只做多/买入/清仓
4,组合构建,择时满仓/空仓,择时满仓/空仓: 均线/买入/清仓
5,回测假设,信号下一期生效；扣成本/滑点,信号下一期生效: 下一交易日；扣成本/滑点: 成本/bps


## 12.8 常见误区

下面这张表比背策略名更重要。每次看到新策略，都先问它有没有说清楚这些问题。


In [8]:
pitfall_table = pd.DataFrame(
    [
        ("只按名字分类", "动量既可以是趋势策略，也可以是多因子里的一个因子。", "看信号和权重生成方式，而不是只看策略名。"),
        ("忘记写仓位方向", "同样是均值回归，多头 ETF 和多空配对交易的可交易约束完全不同。", "明确写出多头、空仓、做空、杠杆和净敞口。"),
        ("漏掉回测假设", "信号当天成交、成本为 0、成分事后可见，都会让结果虚高。", "标签里固定保留成交时点、成本、调仓日和数据可得性。"),
        ("把频率越高当成越高级", "高频更依赖撮合、盘口、延迟和成本模型。", "先用低频策略练好数据、信号、权重、回测和报告闭环。"),
    ],
    columns=["误区", "为什么危险", "更好的检查方式"],
)
pitfall_table


,误区,为什么危险,更好的检查方式
0,只按名字分类,动量既可以是趋势策略，也可以是多因子里的一个因子。,看信号和权重生成方式，而不是只看策略名。
1,忘记写仓位方向,同样是均值回归，多头 ETF 和多空配对交易的可交易约束完全不同。,明确写出多头、空仓、做空、杠杆和净敞口。
2,漏掉回测假设,信号当天成交、成本为 0、成分事后可见，都会让结果虚高。,标签里固定保留成交时点、成本、调仓日和数据可得性。
3,把频率越高当成越高级,高频更依赖撮合、盘口、延迟和成本模型。,先用低频策略练好数据、信号、权重、回测和报告闭环。


## 12.9 保存本章结果并交给下一章

本章不会生成交易信号，只保存分类表。下一章会从这些标签出发，把三种经典策略写成目标权重矩阵，再复用 `lib.backtest`、`lib.evaluation`、`lib.reporting` 和 `lib.trading` 输出结果。


In [9]:
artifacts = {
    "taxonomy_axes": taxonomy_axes,
    "mainline_map": mainline_map,
    "strategy_cards": strategy_cards,
    "exercise_answer": exercise_answer,
    "pitfall_table": pitfall_table,
}

artifact_rows = []
for name, df in artifacts.items():
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    artifact_rows.append({"artifact": name, "path": str(path.relative_to(PROJECT_ROOT)), "rows": len(df)})

artifact_table = pd.DataFrame(artifact_rows)
artifact_table


,artifact,path,rows
0,taxonomy_axes,outputs\results\chapter12_strategy_taxonomy\ta...,6
1,mainline_map,outputs\results\chapter12_strategy_taxonomy\ma...,6
2,strategy_cards,outputs\results\chapter12_strategy_taxonomy\st...,4
3,exercise_answer,outputs\results\chapter12_strategy_taxonomy\ex...,6
4,pitfall_table,outputs\results\chapter12_strategy_taxonomy\pi...,4


In [10]:
handoff_to_13 = strategy_cards.loc[
    strategy_cards["下一章对应"].str.startswith("13"),
    ["策略", "信号来源", "组合构建", "关键回测假设", "下一章对应"],
]
handoff_to_13


,策略,信号来源,组合构建,关键回测假设,下一章对应
1,双均线趋势,时间序列趋势/动量,择时：短均线高于长均线则持有，否则空仓,复权价格；信号下一期生效；震荡市成本敏感,13 策略一
2,布林带均值回归,价格偏离后的均值回归,状态机：跌破下轨买入，回到中轨退出,强趋势会失效；需要成本、止损或持仓天数约束,13 策略二
3,横截面动量 TopN,资产之间的相对强弱,过去收益排名 TopN 等权,统一调仓日；处理缺失值；扣换手成本,13 策略三


## 12.10 小结

现在读者应该能用六个维度给一个策略定位：信号来源、持仓周期、资产范围、仓位方向、组合构建和回测假设。下一章开始写经典策略代码时，不再只问“规则是什么”，还要同步问“它的坐标是什么、权重矩阵怎么生成、哪些假设必须写进回测”。
